In [15]:
import pandas as pd
import numpy as np

In [17]:
df=pd.read_csv(r"C:\Users\rauna\OneDrive\Desktop\MLProject\data\customer_features.csv")
df

,Customer ID,LastPurchaseDate,Frequency,Monetary,AverageOrderValue,Recency,RepeatPurchase_90D,ProductDiversity,AvgItemsPerOrder
0,12346.0,2011-01-18 10:01:00,12,77556.46,6463.038333,234,0,27,6190.416667
1,12347.0,2011-08-02 08:48:00,6,3402.39,567.065000,38,1,107,349.833333
2,12348.0,2011-04-05 10:47:00,4,1709.40,427.350000,157,1,25,624.250000
3,12349.0,2010-10-28 08:23:00,3,2671.14,890.380000,316,1,90,331.000000
4,12350.0,2011-02-02 16:01:00,1,334.40,334.400000,219,0,17,197.000000
...,...,...,...,...,...,...,...,...,...
5276,18283.0,2011-09-05 12:35:00,16,1706.20,106.637500,4,1,285,65.062500
5277,18284.0,2010-10-04 11:33:00,1,461.68,461.680000,340,0,28,494.000000
5278,18285.0,2010-02-17 10:24:00,1,427.00,427.000000,569,0,12,145.000000
5279,18286.0,2010-08-20 11:57:00,2,1296.43,648.215000,385,0,67,304.000000


In [19]:
feature_cols = [
    'Recency',
    'Frequency',
    'Monetary',
    'AverageOrderValue',
    'ProductDiversity',
    'AvgItemsPerOrder'
]

X = df[feature_cols].copy()
y = df['RepeatPurchase_90D'].copy()

print("Features:", X.shape)
print("Target:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())


Features: (5281, 6)
Target: (5281,)

Target distribution:
RepeatPurchase_90D
0    2991
1    2290
Name: count, dtype: int64


In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


Training data: (4224, 6)
Testing data: (1057, 6)


In [23]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.pipeline import Pipeline

# Features requiring log transformation
log_cols = [
    'Frequency',
    'Monetary',
    'AverageOrderValue',
    'ProductDiversity',
    'AvgItemsPerOrder'
]

# Apply log transformation selectively
feature_transformer = ColumnTransformer(
    transformers=[
        ('log', FunctionTransformer(np.log1p), log_cols)
    ],
    remainder='passthrough'
)

# Complete preprocessing pipeline
preprocessor = Pipeline([
    ('feature_transform', feature_transformer),
    ('scaler', StandardScaler())
])

print("Preprocessing pipeline created successfully.")


Preprocessing pipeline created successfully.


In [27]:

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Create Logistic Regression pipeline
logistic_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train the model
logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")


Logistic Regression trained successfully.


In [29]:

from sklearn.tree import DecisionTreeClassifier

# Create Decision Tree model
decision_tree_model = DecisionTreeClassifier(
    random_state=42
)

# Train the model
decision_tree_model.fit(X_train, y_train)

print("Decision Tree trained successfully.")


Decision Tree trained successfully.


In [31]:

from sklearn.ensemble import RandomForestClassifier

# Create Random Forest model
random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# Train the model
random_forest_model.fit(X_train, y_train)

print("Random Forest trained successfully.")


Random Forest trained successfully.


In [33]:

from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

# Create KNN pipeline
knn_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(
        n_neighbors=5
    ))
])

# Train the model
knn_model.fit(X_train, y_train)

print("KNN model trained successfully.")


KNN model trained successfully.


In [35]:

from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

# Create SVM pipeline
svm_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', SVC(
        kernel='rbf',
        random_state=42
    ))
])

# Train the model
svm_model.fit(X_train, y_train)

print("SVM model trained successfully.")


SVM model trained successfully.


In [37]:

import joblib
import os

# Create models directory
model_path = r"C:\Users\rauna\OneDrive\Desktop\MLProject\models"
os.makedirs(model_path, exist_ok=True)

# Models to save
models = {
    'logistic_regression': logistic_model,
    'decision_tree': decision_tree_model,
    'random_forest': random_forest_model,
    'knn': knn_model,
    'svm': svm_model,
    'baseline': baseline_model
}

# Save models
for name, model in models.items():
    joblib.dump(model, os.path.join(model_path, f'{name}.pkl'))

print("All trained models saved successfully.")


All trained models saved successfully.


In [39]:

test_data = X_test.copy()
test_data['RepeatPurchase_90D'] = y_test

test_data.to_csv(
    r"C:\Users\rauna\OneDrive\Desktop\MLProject\data\test_dataset.csv",
    index=False
)

print("Test dataset saved successfully.")
print("Test dataset shape:", test_data.shape)


Test dataset saved successfully.
Test dataset shape: (1057, 7)



# Model Development

## 1. Objective
The objective is to develop supervised machine learning classification models to predict whether an existing customer will make at least one paid purchase within the next 90 days.

The target variable is `RepeatPurchase_90D`:
- `1` – Customer makes a repeat purchase.
- `0` – Customer does not make a repeat purchase.

## 2. Dataset Preparation
- Loaded the engineered customer-level dataset (`customer_features.csv`).
- Separated the dataset into input features (X) and target variable (y).
- Used six customer behaviour features:
  - Recency
  - Frequency
  - Monetary
  - AverageOrderValue
  - ProductDiversity
  - AvgItemsPerOrder

## 3. Train-Test Split
Split the dataset into:
- 80% Training data.
- 20% Testing data.

Used stratified sampling (`stratify=y`) to maintain the target class proportions.

Used `random_state=42` for reproducibility.

## 4. Preprocessing Pipeline
Created a preprocessing pipeline using:
- `FunctionTransformer(np.log1p)` for log transformation of five skewed features.
- `ColumnTransformer` to apply transformations selectively.
- `StandardScaler` to standardize all six features.
- `Pipeline` to combine preprocessing operations.

Preprocessing was incorporated into the scale-sensitive models so that transformations are fitted on training data.

## 5. Baseline Model
Created and trained a `DummyClassifier` using the `most_frequent` strategy.

Purpose:
- Establish a simple baseline.
- Provide a reference for comparing actual machine learning models.

## 6. Machine Learning Model Training

Trained five classification algorithms:

| Algorithm | Approach |
|---|---|
| Logistic Regression | Linear classification |
| Decision Tree | Tree-based classification |
| Random Forest | Ensemble learning |
| K-Nearest Neighbors (KNN) | Distance-based classification |
| Support Vector Machine (SVM) | Margin-based classification |

### Model Configurations
- Logistic Regression: `max_iter=1000`
- Decision Tree: `random_state=42`
- Random Forest: `n_estimators=200`, `random_state=42`, `n_jobs=-1`
- KNN: `n_neighbors=5`
- SVM: `kernel='rbf'`

## 7. Model Saving
Saved all six trained models (including the baseline) using `joblib`.

Saved the models in the `models/` directory as `.pkl` files.

## 8. Test Dataset Saving
Saved the test features and their corresponding actual target labels in:

`data/test_dataset.csv`

This allows the models to be evaluated in a separate notebook without retraining.

**Status:** Initial model development and training completed.



# Next Steps: Model Evaluation

## 1. Load Trained Models and Test Data
- Create a separate notebook: `07_Model_Evaluation.ipynb`.
- Load the saved models from the `models/` directory using `joblib`.
- Load the saved test dataset.
- Separate test features (X_test) and actual labels (y_test).

## 2. Generate Predictions
- Generate predictions using each trained model.
- Obtain predicted probabilities or decision scores wherever supported.
- Keep the test dataset unchanged.

## 3. Evaluate Model Performance
Evaluate all models using suitable classification metrics:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix
- ROC-AUC

Also evaluate the Dummy Classifier as the baseline.

## 4. Model Comparison
- Create a comparison table containing the evaluation metrics.
- Compare all five machine learning algorithms.
- Identify strengths and weaknesses of each model.
- Determine which model is most suitable for repeat-purchase prediction.

## 5. Visual Evaluation
Create visualizations such as:
- Confusion matrices.
- ROC curves.
- Model performance comparison charts.

## 6. Hyperparameter Tuning
- Select promising models based on initial evaluation.
- Apply GridSearchCV or RandomizedSearchCV.
- Tune important model parameters.
- Use cross-validation within the training data.
- Avoid using the test set during tuning.

## 7. Final Model Selection
- Compare tuned models.
- Select the most suitable model based on multiple performance metrics.
- Evaluate the final selected model on the untouched test dataset.

## 8. Feature Interpretation
- Examine feature importance for tree-based models.
- Interpret Logistic Regression coefficients.
- Identify which customer behaviour patterns are associated with repeat purchases.

## 9. Final Deliverables
- Model comparison results.
- Evaluation visualizations.
- Selected trained model.
- Interpretation of model predictions.
- Findings and conclusions for the research report.

**Note:** The current 80:20 random split is an initial evaluation approach. Since this is a time-based prediction problem, chronological validation should also be considered to assess performance on later customer behaviour.
